# DeepSeek Live Runs


## Setup


In [ ]:
# Libraries
import json
import sys
from pathlib import Path
import pandas as pd

In [ ]:
# Project Root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [ ]:
# Pipeline
%load_ext autoreload
%autoreload 2

import backends
import run
import settings
import utils

needs = {'run': ['generate_part', 'join_parts', 'read_batch', 'part_path',
                 'set_aside_replies'],
         'utils': ['api_key', 'read_lines', 'read_table', 'result_path'],
         'backends': ['USAGE', 'spent', 'record_usage', 'call_api'],
         'settings': ['MODELS', 'GENERATION', 'BATCHES_DIR']}
missing = [f'{name}.{attr}' for name, attrs in needs.items()
           for attr in attrs if not hasattr(globals()[name], attr)]
if missing:
    raise SystemExit('Scripts are out of date, missing: ' + ', '.join(missing)
                     + '\nCopy scripts/ from the latest package and restart the kernel.')

utils.make_directories()
pd.set_option('display.max_colwidth', 70)
print('Scripts are current')

## Model


In [ ]:
MODEL = 'deepseek-v4-flash'
PARTS = 5

spec = next(e for e in settings.MODELS.values() if e['id'] == MODEL)
prompts = utils.read_table(settings.PROMPTS_PATH)
wanted = len(prompts) * settings.GENERATION['replicates']
have = len(utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR)))

print(f'Model      {MODEL} on {spec["provider"]}')
print(f'Billed at  ${spec["price"]["input"]}/M input, '
      f'${spec["price"]["output"]}/M output, no batch discount available')
print(f'Cap        {settings.GENERATION["max_tokens"]} tokens, '
      f'temperature {settings.GENERATION["temperature"]}')
print(f'Collected  {have:,} of {wanted:,}, in {PARTS} parts of '
      f'{-(-wanted // PARTS):,}')
print(f'Key found  {bool(utils.api_key(spec["provider"]))}')

## Rerun


In [ ]:
FRESH = False

if FRESH:
    moved = run.set_aside_replies(MODEL)
    print(f'Earlier pass set aside at {moved}' if moved
          else 'Nothing collected yet, so nothing to set aside')
else:
    print('Normal run: only what is missing will be requested')

## Cost


In [ ]:
OUTPUT_TOKENS = 256          # Output Tokens
INPUT_TOKENS = 98            # Output Tokens

left = wanted - have
price = spec['price']
cost = (left * INPUT_TOKENS * price['input']
        + left * OUTPUT_TOKENS * price['output']) / 1e6
print(f'{left:,} calls outstanding at {OUTPUT_TOKENS} output tokens each')
print(f'  Cost  ${cost:,.2f}, about ${cost / PARTS:,.2f} a part')

## Generation


In [ ]:
# Part Runner

totals = {'read': 0, 'failed': 0, 'truncated': 0, 'repeated': 0,
          'blocked': 0, 'input': 0, 'output': 0, 'cost': 0.0}


def run_part(part):
    path, asked, failures = run.generate_part(MODEL, part, PARTS)
    if not asked:
        print(f'Part {part} of {PARTS}: nothing outstanding')
        return
    # Empty Part
    if not path.exists():
        print(f'Part {part} of {PARTS}: all {asked:,} calls failed, nothing '
              f'written. Fix the cause and run this cell again.')
        return

    # Usage Accounting

    backends.USAGE.update(calls=0, input=0, output=0)
    read, failed, truncated, repeated, blocked = run.read_batch(MODEL, path)
    usage, cost = dict(backends.USAGE), backends.spent(MODEL)
    for name, value in [('read', read), ('failed', failed),
                        ('truncated', truncated), ('repeated', repeated),
                        ('blocked', blocked), ('input', usage['input']),
                        ('output', usage['output']), ('cost', cost)]:
        totals[name] += value

    print(f'\nPart {part} of {PARTS}')
    print(f'Read {read:,} replies, {failed} failed, {truncated} truncated, '
          f'{repeated:,} already had')
    print(f'Tokens: {usage["input"]:,} input, {usage["output"]:,} output')
    print(f'Cost: ${cost:,.2f}')
    print(f'Output tokens a reply: {usage["output"] / max(read - failed, 1):.0f}')


print(f'{PARTS} parts of {-(-wanted // PARTS):,}, '
      f'{utils.WORKERS} requests in flight at a time')

In [ ]:
run_part(1)

In [ ]:
run_part(2)

In [ ]:
run_part(3)

In [ ]:
run_part(4)

In [ ]:
run_part(5)

## Merge


In [ ]:
joined, lines = run.join_parts(MODEL, PARTS)
print(f'Joined {lines:,} responses into {joined.name}, part files removed')

print(f'\nAll parts')
print(f'Read {totals["read"]:,} replies, {totals["failed"]} failed, '
      f'{totals["truncated"]} truncated, {totals["repeated"]:,} already had, '
      f'{totals["blocked"]} blocked')
print(f'Tokens: {totals["input"]:,} input, {totals["output"]:,} output')
print(f'Cost: ${totals["cost"]:,.2f}')
print(f'Output tokens a reply: '
      f'{totals["output"] / max(totals["read"] - totals["failed"], 1):.0f}')

## Validation


In [ ]:
# Update Flags

import flags
flags.apply(MODEL)
collected = utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR))
prompts = utils.read_table(settings.PROMPTS_PATH)

if collected.empty:
    print(f'Nothing collected for {MODEL} yet')
else:
    marked = lambda name: collected.get(name, pd.Series('', index=collected.index)) \
        .astype(str).str.strip().isin(['True', 'true']) \
        | ~collected.get(name, pd.Series('', index=collected.index)) \
        .astype(str).str.strip().isin(['', 'False', 'false', 'nan'])
    blank = int((collected['response'].astype(str).str.strip() == '').sum())
    errored = int((collected['error'].astype(str).str.strip() != '').sum())
    print(f'Replies: {len(collected):,}, {blank} empty, {errored} errored, '
          f'{int(marked("blocked").sum())} blocked, '
          f'{int(marked("truncated").sum())} truncated')
    print(f"Coverage: {collected['prompt_id'].nunique():,} of {len(prompts):,} "
          f"prompts")

    display(collected.merge(prompts[['prompt_id', 'condition', 'prompt']],
                            on='prompt_id')[['condition', 'prompt',
                                             'response']].head(5))